# Telco customer churn: logistic-regression coefficients and odds ratios

## Goal
Fit the frozen Pipeline recipe on the **training set** and convert the logistic-regression coefficients into odds ratios (OR), to describe which features go with higher or lower odds of churn.

## Ground rules
- Fitted on the training set only, to interpret coefficients. **The test set is not used**; the frozen threshold and recipe are not changed.
- An OR is an association, not a causal effect, and is not used to go back and choose a model or threshold.
- Numeric columns are standardised: their OR is the effect of **one standard deviation**, not of one more month.
- One-hot encoding keeps every category (no reference level is dropped), so the ORs of one variable's categories are relative to each other and should not be read in isolation.
- `tenure`, `MonthlyCharges` and `TotalCharges` are strongly correlated, so their individual coefficients may be unstable.

## Setup
Load the frozen plan, the unfitted recipe and the raw CSV, and reproduce the same 80/20 split as [06-final-evaluation](06-final-evaluation.ipynb).

In [1]:
from pathlib import Path
import hashlib
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split

# Project folder = the nearest parent that contains reports/final-plan.json
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reports" / "final-plan.json").exists())

plan = json.loads((PROJECT / "reports/final-plan.json").read_text())
assert plan["frozen"] is True
print("Plan frozen:", plan["frozen"], "| fixed threshold (not used in this notebook):", plan["threshold"])

# Unfitted recipe, with its SHA-256 checked against the plan
recipe_path = PROJECT / plan["recipe_artifact"]
assert hashlib.sha256(recipe_path.read_bytes()).hexdigest() == plan["recipe_sha256"], "recipe does not match the frozen record"
recipe = joblib.load(recipe_path)
print("Recipe steps:", list(recipe.named_steps))

Plan frozen: True | fixed threshold (not used in this notebook): 0.409
Recipe steps: ['preprocess', 'model']


In [2]:
data_path = PROJECT / "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(data_path)
assert df["customerID"].is_unique

y = df["Churn"].copy()
x = df.drop(columns=["Churn", "customerID"]).copy()
# The 11 blanks in TotalCharges become NaN and are imputed by the recipe
x["TotalCharges"] = pd.to_numeric(x["TotalCharges"], errors="coerce")

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=2002, stratify=y
)
assert len(x_train) == 5634 and len(x_test) == 1409
print("Train:", len(x_train), "| churn rate", round((y_train == "Yes").mean(), 4))

# The test set is dropped here so that it cannot take part in interpretation or selection
del x_test, y_test

Train: 5634 | churn rate 0.2654


## 1. Fitting on the training set
`clone(recipe)` gives a new unfitted copy, fitted on `x_train`, `y_train` only. The original `recipe` is left as it is.

In [3]:
pipeline = clone(recipe)
pipeline.fit(x_train, y_train)
print("Fitted on", len(x_train), "training customers.")

Fitted on 5634 training customers.


## 2. Feature names and coefficients
- Feature names: `get_feature_names_out()` of the preprocessing step (the column names after one-hot encoding).
- Coefficients: `coef_[0]` of the model step. `classes_` gives the direction: the coefficients refer to the odds of `Yes`.

In [4]:
# 46 features after encoding: 3 numeric (numeric__) and 43 categorical (categorical__)
feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
# coef_ has shape (1, 46); [0] gives a 1-D array aligned with feature_names
coefs = pipeline.named_steps["model"].coef_[0]

# classes_ is ['No' 'Yes']: positive = higher odds of churn, negative = lower
print(pipeline.named_steps["model"].classes_)
print(len(feature_names), len(coefs))

['No' 'Yes']
46 46


## 3. Odds ratios
`OR = exp(coefficient)`. A coefficient of 0 gives OR 1 (no association); a positive one gives OR > 1 (higher odds of churn); a negative one gives OR < 1. The intercept is a baseline and is left out of the table.

In [5]:
odds_ratio = np.exp(coefs)

odds = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs,
    "odds_ratio": odds_ratio,
})

# Largest OR first
odds = odds.sort_values("odds_ratio", ascending=False)
print(odds.shape)
display(odds)

(46, 3)


,feature,coef,odds_ratio
2,numeric__TotalCharges,0.734812,2.085089
37,categorical__Contract_Month-to-month,0.569062,1.766610
17,categorical__InternetService_Fiber optic,0.268608,1.308143
44,categorical__PaymentMethod_Electronic check,0.201875,1.223695
1,numeric__MonthlyCharges,0.157267,1.170308
28,categorical__TechSupport_No,0.154725,1.167337
19,categorical__OnlineSecurity_No,0.150970,1.162962
14,categorical__MultipleLines_No phone service,0.076648,1.079662
11,categorical__PhoneService_No,0.076648,1.079662
22,categorical__OnlineBackup_No,0.034760,1.035371


## 4. Reading the result
The five largest and five smallest odds ratios.

In [6]:
print(odds.head())
print(odds.tail())

                                        feature      coef  odds_ratio
2                         numeric__TotalCharges  0.734812    2.085089
37         categorical__Contract_Month-to-month  0.569062    1.766610
17     categorical__InternetService_Fiber optic  0.268608    1.308143
44  categorical__PaymentMethod_Electronic check  0.201875    1.223695
1                       numeric__MonthlyCharges  0.157267    1.170308
                             feature      coef  odds_ratio
40  categorical__PaperlessBilling_No -0.294175    0.745146
12     categorical__PhoneService_Yes -0.336503    0.714264
16  categorical__InternetService_DSL -0.384833    0.680565
39    categorical__Contract_Two year -0.768277    0.463811
0                    numeric__tenure -1.521221    0.218445


### How much is "one standard deviation"?
Numeric columns are standardised in the recipe as `(value − mean) / standard deviation`, so their OR is the effect of one standard deviation. The mean and standard deviation are learned by `StandardScaler` from the training set and stored in `mean_` and `scale_`.

In [7]:
# Pipeline -> ColumnTransformer -> numeric route -> scaler
scaler = pipeline.named_steps["preprocess"].named_transformers_["numeric"].named_steps["scaler"]

# Same order as the numeric columns in the plan: tenure, MonthlyCharges, TotalCharges
scale_table = pd.DataFrame({
    "feature": plan["features"]["numeric"],
    "mean": scaler.mean_,
    "std": scaler.scale_,
})
display(scale_table)

,feature,mean,std
0,tenure,32.172346,24.572329
1,MonthlyCharges,64.710774,30.126739
2,TotalCharges,2266.704473,2264.873816


### Interpretation
All odds ratios below come from a logistic regression fitted on the training set. They are **associations, not causal effects**.

- **Contract type**: `Month-to-month` OR 1.77, `Two year` OR 0.46. Categories of one variable are compared with each other: 1.77 / 0.46 ≈ 3.8, so with the other features held constant the odds of churn for a month-to-month customer are about 3.8 times those of a two-year customer. The direction agrees with the descriptive rates in notebook 02 (42.71% against 2.83%).
- **Internet service**: `Fiber optic` OR 1.31, `DSL` OR 0.68, a ratio of about 1.9 (descriptive rates: 41.89% against 18.96%).
- **Tenure**: `tenure` has OR 0.22, the smallest in the table. Being standardised, it means that one more standard deviation of tenure (about 24.6 months, see the table above) goes with odds of churn about 0.22 times as high. It is not the effect of one more month.
- **Payment method**: `Electronic check` OR 1.22, in line with its higher descriptive churn rate (45.29%).
- **`TotalCharges` is not interpreted on its own** (OR 2.09, the largest). It is strongly correlated with `tenure` and has the opposite sign (0.73 against −1.52), a sign of collinearity. The single coefficient is unstable and does not show that customers who have paid more in total are more likely to leave.
- **No reference level is dropped**: categories of one variable are relative to each other. `PhoneService_No` and `MultipleLines_No phone service` have exactly the same coefficient (0.076648), which shows that the two columns are near duplicates and should not be read separately.
- The logistic regression uses the default L2 penalty (C = 1.0), which shrinks coefficients towards 0, so the exact OR values should not be quoted with too much precision.

## Checks
- Number of coefficients equals number of features after preprocessing: 46 = 46.
- The OR of numeric columns is described per standard deviation, with the standard deviations shown: tenure 24.57 months, MonthlyCharges 30.13.
- The test set is not used: it is deleted in the setup cell and never referenced again. The recipe and the threshold are unchanged.